In [ ]:
# ==============================================================
# PROYECTO: Predicción de ley de Cu en yacimiento tipo veta (Chile)
# Interpolación espacial vecinal con Transformer + prior Matérn
# ==============================================================
# NOTEBOOK DE LIMPIEZA — carga el dataset CRUDO, lo analiza,
# lo limpia y guarda 'sondajes_clean.xlsx'
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, warnings, os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree
warnings.filterwarnings('ignore')

# ---- Carga del repositorio -----------------------------------
# Si se ejecuta en Colab (o entorno sin el archivo), clona el repo.
REPO_URL = 'https://github.com/ARSENICOX99/Proyecto-final-RNP.git'
REPO_DIR = 'Proyecto-final-RNP'

if not os.path.exists('sondajes.xlsx'):
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL}')
    os.chdir(REPO_DIR)

# ---- Rutas relativas -----------------------------------------
RUTA_RAW   = 'sondajes.xlsx'          # dataset CRUDO (entrada)
RUTA_CLEAN = 'sondajes_clean.xlsx'    # dataset limpio (salida, celda 10)

# ---- Reproducibilidad ----------------------------------------
SEED = 42
np.random.seed(SEED)

# ---- Columnas del dataset ------------------------------------
COORDS       = ['X', 'Y', 'Z']          # metros reales (para distancias/kernel)
FEATURES_NUM = ['DIP', 'AZIMUT', 'FROM'] # numéricas contextuales -> scaler
FEATURE_CAT  = 'LITOLOGIA'               # categórica -> embedding
TARGET       = 'CUT'                     # ley de cobre (%)
HOLE         = 'HOLE-ID'

# ---- Carga del crudo -----------------------------------------
df = pd.read_excel(RUTA_RAW)
df = df.sort_values([HOLE, 'FROM']).reset_index(drop=True)

print(f"Dataset CRUDO cargado: {df.shape[0]:,} intervalos | {df[HOLE].nunique()} sondajes")
print(f"Columnas: {list(df.columns)}")
print(f"\nCUT (ley Cu %):  min={df[TARGET].min():.3f}  "
      f"mediana={df[TARGET].median():.3f}  max={df[TARGET].max():.3f}  "
      f"skew={df[TARGET].skew():.2f}")

In [ ]:
# ── 1. ESTADÍSTICA DESCRIPTIVA BÁSICA ─────────────────────────
print("\n" + "=" * 55)
print("ESTADÍSTICA DESCRIPTIVA")
print("=" * 55)
print(df.describe().T.round(4))

print("\n--- Valores nulos por columna ---")
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)
print(pd.DataFrame({'nulos': nulos, '%': pct}))

In [ ]:
# ── 2. DISTRIBUCIÓN DE CUT ────────────────────────────────────
p99 = df['CUT'].quantile(0.99)
p995 = df['CUT'].quantile(0.995)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Distribución de la Ley de Cobre (CUT)', fontweight='bold')

# Histograma completo
axes[0].hist(df['CUT'], bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('Distribución completa')
axes[0].set_xlabel('CUT (%)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(p99, color='red', linestyle='--', label=f'p99 = {p99:.2f}')
axes[0].legend()

# Sin outliers extremos (hasta p99)
cut_clip = df['CUT'].clip(upper=p99)
axes[1].hist(cut_clip, bins=80, color='teal', edgecolor='none')
axes[1].set_title(f'Cappada en p99 ({p99:.2f}%)')
axes[1].set_xlabel('CUT (%)')

# Boxplot
axes[2].boxplot(df['CUT'].clip(upper=p99), vert=True,
                patch_artist=True,
                boxprops=dict(facecolor='lightsteelblue'))
axes[2].set_title('Boxplot (sin outliers extremos)')
axes[2].set_ylabel('CUT (%)')

plt.tight_layout()
plt.show()

print(f"\n--- Outliers en CUT ---")
print(f"  Valores > p99  ({p99:.2f}%)  : {(df['CUT'] > p99).sum():,} registros")
print(f"  Valores > p99.5 ({p995:.2f}%) : {(df['CUT'] > p995).sum():,} registros")
print(f"  Valor máximo               : {df['CUT'].max():.2f}%")
print(f"  → Se recomienda cappear en p99 = {p99:.2f}%")


In [ ]:
# ── 3. LONGITUD DE SONDAJES ───────────────────────────────────
longitudes = df.groupby('HOLE-ID').size().reset_index(name='n_intervalos')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Longitud de Sondajes', fontweight='bold')

axes[0].hist(longitudes['n_intervalos'], bins=50, color='darkorange', edgecolor='none')
axes[0].set_xlabel('Nº de intervalos por sondaje')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de longitudes')

axes[1].boxplot(longitudes['n_intervalos'], vert=False,
                patch_artist=True,
                boxprops=dict(facecolor='moccasin'))
axes[1].set_xlabel('Nº de intervalos')
axes[1].set_title('Boxplot')

plt.tight_layout()
plt.show()

print(f"\n--- Estadísticas de longitud de sondajes ---")
print(longitudes['n_intervalos'].describe().round(1))
print(f"  Sondajes con < 50 intervalos  : {(longitudes['n_intervalos'] < 50).sum()}")
print(f"  Sondajes con > 200 intervalos : {(longitudes['n_intervalos'] > 200).sum()}")


In [ ]:
# ── 4. DISTRIBUCIÓN ESPACIAL ──────────────────────────────────
centroides = df.groupby('HOLE-ID').agg(
    X_mean=('X', 'mean'), Y_mean=('Y', 'mean'),
    Z_mean=('Z', 'mean'), CUT_mean=('CUT', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución Espacial de Sondajes', fontweight='bold')

sc = axes[0].scatter(centroides['X_mean'], centroides['Y_mean'],
                     c=centroides['CUT_mean'], cmap='YlOrRd',
                     s=20, alpha=0.8)
plt.colorbar(sc, ax=axes[0], label='Ley media CUT (%)')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Planta (X vs Y) — color = ley media')

sc2 = axes[1].scatter(centroides['X_mean'], centroides['Z_mean'],
                      c=centroides['CUT_mean'], cmap='YlOrRd',
                      s=20, alpha=0.8)
plt.colorbar(sc2, ax=axes[1], label='Ley media CUT (%)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Z (elevación)')
axes[1].set_title('Sección (X vs Z) — color = ley media')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5. PERFILES DE LEY (muestra de sondajes) ──────────────────
N_MUESTRAS = 12
ids_muestra = longitudes[longitudes['n_intervalos'] >= 50].sample(N_MUESTRAS, random_state=42)['HOLE-ID']

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
fig.suptitle('Perfiles de Ley por Sondaje (muestra)', fontweight='bold')
axes = axes.flatten()

for i, hole_id in enumerate(ids_muestra):
    sondaje = df[df['HOLE-ID'] == hole_id].sort_values('FROM')
    axes[i].plot(sondaje['FROM'], sondaje['CUT'].clip(upper=p99),
                 color='steelblue', linewidth=0.8)
    axes[i].set_title(hole_id, fontsize=9)
    axes[i].set_xlabel('Prof. (m)', fontsize=8)
    axes[i].set_ylabel('CUT (%)', fontsize=8)
    axes[i].tick_params(labelsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# ── 6. ANÁLISIS DE LITOLOGÍA ──────────────────────────────────
print("\n" + "=" * 55)
print("ANÁLISIS DE LITOLOGÍA")
print("=" * 55)

total = len(df)
nulos_lito = df['LITOLOGIA'].isnull().sum()
print(f"  Nulos en LITOLOGÍA : {nulos_lito:,} ({nulos_lito/total*100:.1f}%)")
print(f"  Con litología      : {total - nulos_lito:,}")
print()
print("  Frecuencia por clase:")
print(df['LITOLOGIA'].value_counts().to_frame('count').assign(
    pct=lambda x: (x['count'] / total * 100).round(2)
))

# Ley media por litología
print("\n  Ley media CUT por clase litológica:")
print(df.groupby('LITOLOGIA')['CUT'].mean().round(4).sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Litología', fontweight='bold')

df['LITOLOGIA'].value_counts().plot(kind='bar', ax=axes[0], color='slateblue')
axes[0].set_title('Frecuencia por clase')
axes[0].set_xlabel('Clase litológica')
axes[0].set_ylabel('Nº de intervalos')

df.boxplot(column='CUT', by='LITOLOGIA', ax=axes[1])
axes[1].set_title('CUT por clase litológica')
axes[1].set_xlabel('Clase litológica')
axes[1].set_ylabel('CUT (%)')
plt.suptitle('')

plt.tight_layout()
plt.show()


In [ ]:
# ── 7. CORRELACIONES ──────────────────────────────────────────
cols_num = ['CUT', 'X', 'Y', 'Z', 'DIP', 'AZIMUT', 'FROM', 'LITOLOGIA']
corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True)
ax.set_title('Matriz de Correlación', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. LIMPIEZA Y CAPPEO ──────────────────────────────────────
print("\n" + "=" * 55)
print("LIMPIEZA")
print("=" * 55)

df_clean = df.copy()

# --- ELIMINAR DUPLICADOS (primero, antes de todo) ---
n_antes = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
n_dup = n_antes - len(df_clean)
print(f"  Duplicados exactos eliminados: {n_dup:,}")
print(f"  Registros: {n_antes:,} -> {len(df_clean):,}")

# Verificar que no queden intervalos (HOLE-ID, FROM, TO) repetidos con distinto valor
resto = df_clean.duplicated(subset=['HOLE-ID', 'FROM', 'TO']).sum()
print(f"  Intervalos (HOLE-ID,FROM,TO) aún repetidos: {resto}")
if resto > 0:
    print(f"  ⚠ Revisar: hay mismos intervalos con valores distintos")

# --- Cappeo CUT en p99 ---
UMBRAL_CUT = p99
n_caps = (df_clean['CUT'] > UMBRAL_CUT).sum()
df_clean['CUT'] = df_clean['CUT'].clip(upper=UMBRAL_CUT)
print(f"  CUT cappeada en {UMBRAL_CUT:.4f}% → {n_caps:,} valores modificados")

# --- Eliminar sondajes muy cortos ---
LARGO_MIN = 20
sondajes_cortos = longitudes[longitudes['n_intervalos'] < LARGO_MIN]['HOLE-ID']
df_clean = df_clean[~df_clean['HOLE-ID'].isin(sondajes_cortos)]
print(f"  Sondajes eliminados (< {LARGO_MIN} intervalos): {len(sondajes_cortos)}")
print(f"  Registros restantes: {len(df_clean):,}")

In [ ]:
# ── 9. IMPUTACIÓN DE LITOLOGÍA (ffill/bfill + KNN espacial 3D) ──
from scipy.stats import mode as scipy_mode
from scipy.spatial import cKDTree

print("\n" + "=" * 55)
print("IMPUTACIÓN DE LITOLOGÍA")
print("=" * 55)

df_clean = df_clean.sort_values(['HOLE-ID', 'FROM']).reset_index(drop=True)
nulos_ini = df_clean['LITOLOGIA'].isnull().sum()

# --- Paso 1: ffill/bfill dentro de cada sondaje (imputación vertical) ---
# Rellena sondajes con litología PARCIAL usando continuidad a lo largo del pozo
df_clean['LITOLOGIA'] = (
    df_clean.groupby('HOLE-ID')['LITOLOGIA']
    .transform(lambda s: s.ffill().bfill())
)
nulos_post_ffill = df_clean['LITOLOGIA'].isnull().sum()
print(f"  Nulos iniciales           : {nulos_ini:,}")
print(f"  Tras ffill/bfill vertical : {nulos_post_ffill:,}  "
      f"(resueltos {nulos_ini - nulos_post_ffill:,})")

# --- Paso 2: KNN espacial 3D para sondajes 100% sin litología ---
# Flag para trazabilidad (documentar qué se imputó espacialmente)
df_clean['LITO_IMPUTADA'] = df_clean['LITOLOGIA'].isnull()

conocidos = df_clean[df_clean['LITOLOGIA'].notnull()]
faltantes = df_clean[df_clean['LITOLOGIA'].isnull()]

if len(faltantes) > 0:
    # Árbol solo con intervalos de litología conocida, usando SOLO coordenadas
    # (NO usar CUT como predictor -> evita circularidad lito<->ley)
    tree = cKDTree(conocidos[['X', 'Y', 'Z']].values)
    K_VECINOS = 5
    _, idx = tree.query(faltantes[['X', 'Y', 'Z']].values, k=K_VECINOS)

    lito_conocida = conocidos['LITOLOGIA'].values
    # Voto por mayoría entre los K vecinos más cercanos
    vecinos_lito = lito_conocida[idx]                       # (n_faltantes, K)
    imputadas = scipy_mode(vecinos_lito, axis=1, keepdims=False).mode

    df_clean.loc[faltantes.index, 'LITOLOGIA'] = imputadas

print(f"  Imputados por KNN 3D      : {len(faltantes):,}")
print(f"  Nulos finales             : {df_clean['LITOLOGIA'].isnull().sum()}")

df_clean['LITOLOGIA'] = df_clean['LITOLOGIA'].astype(int)

print(f"\n  Distribución final de LITOLOGÍA:")
print(df_clean['LITOLOGIA'].value_counts().sort_index())
print(f"\n  Intervalos imputados espacialmente: "
      f"{df_clean['LITO_IMPUTADA'].sum():,} "
      f"({100*df_clean['LITO_IMPUTADA'].mean():.1f}%)")

In [ ]:
# ── 10. GUARDAR DATASET LIMPIO ────────────────────────────────
RUTA_CLEAN = 'sondajes_clean.xlsx'   # se guarda en la carpeta del repo/notebook
df_clean.to_excel(RUTA_CLEAN, index=False)
print(f"\n  ✓ Dataset limpio guardado en: {RUTA_CLEAN}")
print(f"  Shape final: {df_clean.shape}")